In [1]:
import torch
import torchvision.models as models
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import cv2

ModuleNotFoundError: No module named 'cv2'

In [40]:
device = torch.device("cuda")

# Load the pretrained ResNet model
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Modify the architecture of the last layer for cat-dog classification
# %%%%%%% TODO 1
num_classes = 2  # Number of classes (cat and dog)
num_features = resnet.fc.in_features
resnet.fc = torch.nn.Linear(num_features, num_classes)

resnet = resnet.to(device)
# # Print the modified ResNet model
# print(resnet)

In [41]:
# Define the transforms for data augmentation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Define the transform to convert binary labels to class probabilities
binary_to_prob_transform = transforms.Lambda(lambda label: torch.Tensor([1.0 - label, label]))

# Define the path to the dataset directory
# &&&&&&& TODO  2
dataset_path = 'datasets/PetImages/'

# Load the dataset
dataset = datasets.ImageFolder(dataset_path, transform=transform)

# Print the classes in the dataset
print(dataset.classes)

['Cat', 'Dog']


In [42]:
# Define the percentage of data to use for validation
val_percentage = 0.2

# Calculate the number of samples for validation
val_size = int(val_percentage * len(dataset))
train_size = len(dataset) - val_size

# Split the dataset into training and validation sets
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# Create the dataloaders for training and validation sets
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [43]:
# Print the number of samples in the training set
print("Number of samples in the training set:", len(train_dataset))

# Print the number of samples in the validation set
print("Number of samples in the validation set:", len(val_dataset))

# Print the number of batches in the training dataloader
print("Number of batches in the training dataloader:", len(train_dataloader))

# Print the number of batches in the validation dataloader
print("Number of batches in the validation dataloader:", len(val_dataloader))

Number of samples in the training set: 19996
Number of samples in the validation set: 4999
Number of batches in the training dataloader: 625
Number of batches in the validation dataloader: 157


In [44]:
# Define the loss function
loss_fn = torch.nn.MSELoss()
lr = 1e-3  # Learning rate
optimizer = torch.optim.Adam(resnet.parameters(), lr=lr)  # Adam optimizer

In [49]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # print(type(X))
        # Compute prediction and loss
        X = X.to(device)
        y = y.to(device)
        pred = model(X)
        print(pred.shape, y.shape)
        loss = loss_fn(y, pred)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y.argmax(1)).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [50]:
epoch = 10
for t in range(epoch):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, resnet, loss_fn, optimizer)
    test_loop(val_dataloader, resnet, loss_fn)

Epoch 1
-------------------------------
torch.Size([32, 2]) torch.Size([32])


/Users/<USER>/miniconda3/envs/torch/lib/python3.11/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([32, 2])) that is different to the input size (torch.Size([32])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


RuntimeError: The size of tensor a (32) must match the size of tensor b (2) at non-singleton dimension 1

In [ ]:
torch.save(resnet.state_dict(), "model.pth")

In [ ]:
# Export to ONNX
dummy_input = torch.randn(1, 3, 224, 224).to(device)
torch.onnx.export(resnet, dummy_input, "model.onnx")